# Proposed validation — review before it runs

**What this measures:** The target metric is the summed absolute weight change over every VLM-side parameter after 8 hand-built Adam(lr=1e-3, wd=0) steps on one fixed random batch — exactly 0.0 if and only if `stop_vlm`'s `jax.lax.stop_gradient` on expert-0 q/k/v severs every expert-loss→trunk gradient path — while floors on VLM movement under `full` (coupling exists), expert movement in both arms, and loss drops in both arms complete the user's pass rule. The whole comparison lives inside one run (both routings trained from a bit-identical `jax.random.key(0)` init on the head revision) because the base branch cannot construct `gradient_routing` at all, so this is a `capability` question with `baseline: none` and no stand-in number is ever emitted.

**Target metric:** `vlm_weight_l1_delta_stop_vlm`

Remyx wrote this test for the change in this PR. **Nothing here has been executed** — there are no outputs, and no result is being claimed.

Edit it if the measurement is wrong, then mention `@remyx validate` again and it will run what you committed. If anything is missing at run time — an import, a dependency, a device — the run reports it and repairs what it can rather than failing silently.

This notebook is the only copy of the test: `.remyx/validation.yaml` points here, and the run executes these cells top to bottom.

In [ ]:
# papermill parameters — Remyx injects variant / ref / seed here
variant = ""
ref = ""
seed = 0

## Execution context

The cells below are the test Remyx drafted, split into cells; it was written as a script, so `__file__` is set to `eval/eval_gradient_routing.py` (no such file is committed). This cell gives it what the command line would: its own path in `__file__`, an empty argument list so `argparse` sees no stray flags, and the papermill parameters as `REMYX_VARIANT` / `REMYX_REF` / `REMYX_SEED` for anything that wants them.

In [ ]:
import os, sys
ROOT = os.getcwd()  # the notebook runs with the repository root as its working directory
__file__ = os.path.join(ROOT, "eval/eval_gradient_routing.py")
sys.argv = [__file__]
for _k in ("variant", "ref", "seed"):
    _v = globals().get(_k)
    if _v not in (None, ""):
        os.environ["REMYX_" + _k.upper()] = str(_v)
print("[remyx] cwd", ROOT, "| script", __file__)

In [ ]:
"""Eval: with gradient_routing='stop_vlm', does the VLM stop learning from the action expert
while the action expert keeps learning? (arXiv:2611.05249 insulation factor, dummy-scale probe)

Trains the dummy pi0 (both `dummy` variants, as in the PR's own tests) for 8 Adam steps on ONE
fixed random batch, twice from bit-identical init: `full` vs `stop_vlm`. Reports per-side
weight L1 movement (VLM vs action expert) and flow-matching loss before/after. The whole
comparison lives inside this single run (capability question, baseline: none). CPU only, no
datasets/checkpoints/network.

Baseline safety: the gradient_routing / action_token_supervision kwargs are added by the PR, so
they are applied only when present in Pi0Config's fields. On the pre-change baseline both arms
fall back to the default full-coupling behaviour (no crash): the arms then run identical
pre-change code, the target metric is degraded (nonzero VLM movement under the "stop_vlm" arm)
and the eval fails, as it should. On the PR head the two arms execute genuinely different
gradient paths.

Expected magnitudes (validation.yaml floors): under stop_vlm every VLM-side weight must stay
bit-identical (Adam with zero grad and NO weight decay updates by exactly 0.0), so the target
is exactly 0.0; under full routing >= 8192 trunk kv coords move (~1e-3 each) => L1 >> 1.024;
the expert (>= 40960 attention coords) moves >> 5.12 under both routings; the flow-matching
loss starts ~ Var(noise)+Var(U(-1,1)) = 4/3 and must drop by > 1.5e-7 (float32-resolvable).
"""

In [ ]:
import argparse
import dataclasses
import json
import os
import sys
from pathlib import Path

In [ ]:
ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))

import jax

jax.config.update("jax_platform_name", "cpu")  # CPU-only compute; never measure CUDA quantities

In [ ]:
import flax.nnx as nnx
import jax.numpy as jnp
import optax

import openpi.models.model as _model
import openpi.models.pi0_config as _pi0_config

In [ ]:
LR = 1e-3
STEPS = 2 if os.environ.get("REMYX_SMOKE") == "1" else 8
N_EVAL_KEYS = 4

# Fields the PR adds to Pi0Config. Absent on the baseline, where the knobs must NOT be passed
# (unknown kwarg -> TypeError); presence is detected defensively via the dataclass fields, which
# are part of the pre-change module, so no import of added symbols is needed.
_PI0_CONFIG_FIELDS = {f.name for f in dataclasses.fields(_pi0_config.Pi0Config)}
_HAS_GRADIENT_ROUTING = "gradient_routing" in _PI0_CONFIG_FIELDS
_HAS_ACTION_TOKEN_SUPERVISION = "action_token_supervision" in _PI0_CONFIG_FIELDS

In [ ]:
# One fixed random batch, keys 1-4 exactly as in src/openpi/models/action_token_supervision_test.py.
OBSERVATION = _model.Observation(
    images={
        key: jax.random.uniform(jax.random.key(1), (2, *_model.IMAGE_RESOLUTION, 3), minval=-1, maxval=1)
        for key in _model.IMAGE_KEYS
    },
    image_masks={key: jnp.ones((2,), dtype=jnp.bool_) for key in _model.IMAGE_KEYS},
    state=jax.random.normal(jax.random.key(2), (2, 8)),
    tokenized_prompt=jax.random.randint(jax.random.key(3), (2, 16), 0, 200_000),
    tokenized_prompt_mask=jnp.ones((2, 16), dtype=jnp.bool_),
)
ACTIONS = jax.random.uniform(jax.random.key(4), (2, 4, 8), minval=-1, maxval=1)

In [ ]:
def _build(routing: str):
    kwargs = dict(
        paligemma_variant="dummy",
        action_expert_variant="dummy",
        action_dim=8,
        action_horizon=4,
        max_token_len=16,
        dtype="float32",
    )
    if _HAS_GRADIENT_ROUTING:
        # Knob added by the PR; this is the in-script enablement of the changed code path. On the
        # baseline the field does not exist and the config falls back to pre-change (full) routing.
        kwargs["gradient_routing"] = routing
    if _HAS_ACTION_TOKEN_SUPERVISION:
        kwargs["action_token_supervision"] = False  # no aux CE term, so the exact-zero VLM criterion is meaningful
    config = _pi0_config.Pi0Config(**kwargs)
    return config.create(jax.random.key(0))  # bit-identical init across arms

In [ ]:
def _flat_params(model) -> dict[str, jax.Array]:
    params = nnx.state(model, nnx.Param)
    return {"/".join(map(str, path)): var.value for path, var in params.flat_state().items()}

def _is_expert(path: str) -> bool:
    # Repo test convention (action_token_supervision_test.py): expert-1 submodules carry a "_1"
    # name suffix; pi0's own action projections are expert-side. Everything else is VLM-side.
    return "_1" in path or "action_in_proj" in path or "action_out_proj" in path

In [ ]:
def _eval_loss(model) -> float:
    keys = [jax.random.fold_in(jax.random.key(101), i) for i in range(N_EVAL_KEYS)]
    losses = [jnp.mean(model.compute_loss(key, OBSERVATION, ACTIONS)) for key in keys]
    return float(jnp.mean(jnp.asarray(losses)))

@nnx.jit
def _train_step(model, optimizer, rng):
    # optax.adam (NOT adamw) applies NO weight decay: zero-grad params get exactly 0.0 update,
    # so VLM weights under stop_vlm stay bit-identical instead of decaying toward 0.
    loss, grads = nnx.value_and_grad(lambda m: jnp.mean(m.compute_loss(rng, OBSERVATION, ACTIONS)))(model)
    optimizer.update(grads)
    return loss

In [ ]:
def _run_arm(routing: str) -> dict[str, float]:
    model = _build(routing)
    before = _flat_params(model)
    loss_before = _eval_loss(model)

    optimizer = nnx.Optimizer(model, optax.adam(learning_rate=LR, b1=0.9, b2=0.999, eps=1e-8))
    for step in range(STEPS):  # identical per-step noise/timestep keys across arms
        _train_step(model, optimizer, jax.random.fold_in(jax.random.key(7), step))

    after = _flat_params(model)
    vlm_delta = sum(float(jnp.abs(after[p] - before[p]).sum()) for p in before if not _is_expert(p))
    expert_delta = sum(float(jnp.abs(after[p] - before[p]).sum()) for p in before if _is_expert(p))
    return {
        "vlm_delta": vlm_delta,
        "expert_delta": expert_delta,
        "vlm_count": int(sum(before[p].size for p in before if not _is_expert(p))),
        "expert_count": int(sum(before[p].size for p in before if _is_expert(p))),
        "loss_before": loss_before,
        "loss_after": _eval_loss(model),
    }

In [ ]:
def main():
    # Accepted and ignored per the output contract.
    parser = argparse.ArgumentParser()
    parser.add_argument("--variant", default=None)
    parser.add_argument("--ref", default=None)
    parser.add_argument("--seed", type=int, default=None)
    parser.parse_args()

    full = _run_arm("full")
    stop = _run_arm("stop_vlm")

    metrics = {
        "vlm_weight_l1_delta_stop_vlm": stop["vlm_delta"],
        "vlm_weight_l1_delta_full": full["vlm_delta"],
        "expert_weight_l1_delta_stop_vlm": stop["expert_delta"],
        "expert_weight_l1_delta_full": full["expert_delta"],
        "loss_drop_full": full["loss_before"] - full["loss_after"],
        "loss_drop_stop_vlm": stop["loss_before"] - stop["loss_after"],
        "loss_step0_gap": abs(full["loss_before"] - stop["loss_before"]),
        "loss_before_full": full["loss_before"],
        "loss_after_full": full["loss_after"],
        "loss_before_stop_vlm": stop["loss_before"],
        "loss_after_stop_vlm": stop["loss_after"],
        "vlm_param_count": full["vlm_count"],
        "expert_param_count": full["expert_count"],
    }
    print(json.dumps(metrics))

if __name__ == "__main__":
    main()

## The criteria this is judged against

From `.remyx/validation.yaml` — thresholds live here, not in the test, so a failing measurement reports rather than crashes.

```yaml
question:
  kind: capability
  ask: "With gradient_routing=stop_vlm, does the VLM stop learning from the action expert while the action expert keeps learning?"
loop:
  max_iterations: 8
  fix_code: true
benchmarks:
  - name: pi0-gradient-routing-insulation
    suite: "eval/eval_gradient_routing.py"
    scorer: vlm_weight_l1_delta_stop_vlm
    baseline: none
    policy:
      guardrail_veto: true
    metrics:
      - name: vlm_weight_l1_delta_stop_vlm
        direction: min
        threshold: 0.0
        role: target
        bar: goal
        reads_as: "Sum of |weight change| over all VLM-side params after 8 Adam(wd=0) steps under stop_vlm; pass requires exactly 0.0"
      - name: vlm_weight_l1_delta_full
        direction: max
        threshold: 1.024  # floor: 8192 trunk kv-projection coords (2*1*64*16*4) x lr 1e-3 / worst-case 1-of-8 step retention
        role: guardrail
        bar: floor
        reads_as: "Same VLM-side L1 movement under default full routing; nonzero proves the expert loss couples into the trunk"
      - name: expert_weight_l1_delta_stop_vlm
        direction: max
        threshold: 5.12  # floor: 40960 expert attention coords (10240 x 4 blocks) x lr 1e-3 / 8
        role: guardrail
        bar: floor
        reads_as: "L1 movement of action-expert params (path contains _1 or action_in/out_proj) under stop_vlm; expert must keep training"
      - name: expert_weight_l1_delta_full
        direction: max
        threshold: 5.12  # same derivation as above; the expert trains under both routings
        role: guardrail
        bar: floor
        reads_as: "L1 movement of action-expert params under full routing; the expert moves in both arms per the pass rule"
      - name: loss_drop_full
        direction: max
        threshold: 0.00000015  # floor: float32-resolvable drop = eps * loss0 = 1.19e-7 * 1.33
        role: guardrail
        bar: floor
        reads_as: "Flow-matching loss decrease (before minus after, fixed eval keys) over 8 steps under full routing"
      - name: loss_drop_stop_vlm
        direction: max
        threshold: 0.00000015  # same float32-resolution floor; insulation must not stall the loss
        role: guardrail
        bar: floor
        reads_as: "Flow-matching loss decrease under stop_vlm; the loss must still fall when the trunk is insulated"
      - name: loss_step0_gap
        direction: min
        threshold: 0.0
        role: guardrail
        bar: sanity
        reads_as: "Absolute gap between the two routings' initial losses; stop_gradient is forward-identity so it must be exactly 0"
      - name: loss_before_full
        direction: max
        threshold: 0.5  # flow-matching loss at init ~ Var(noise)+Var(U(-1,1)) = 4/3
        role: diagnostic
        bar: sanity
        reads_as: "Mean flow-matching loss at the shared init, full routing, averaged over 4 fixed eval keys"
      - name: loss_after_full
        direction: max
        threshold: 0.5  # 8 steps at lr=1e-3 leave the loss O(1); catches divergence
        role: diagnostic
        bar: sanity
        reads_as: "Mean flow-matching loss after 8 steps, full routing, same eval keys"
      - name: loss_before_stop_vlm
        direction: max
        threshold: 0.5  # identical to full arm's initial loss by forward-identity
        role: diagnostic
        bar: sanity
        reads_as: "Mean flow-matching loss at the shared init, stop_vlm routing"
      - name: loss_after_stop_vlm
        direction: max
        threshold: 0.5
        role: diagnostic
        bar: sanity
        reads_as: "Mean flow-matching loss after 8 steps, stop_vlm routing, same eval keys"
      - name: vlm_param_count
        direction: max
        threshold: 16000000  # embedder 257152*64 = 16.46M + 4 trunk blocks + tower >= 16.56M
        role: diagnostic
        bar: sanity
        reads_as: "VLM-side parameter coordinate count in the partition; verifies the VLM bucket is the big one"
      - name: expert_param_count
        direction: max
        threshold: 100000  # 4 x 26752 expert-block coords + ~2k action in/out proj coords ~ 109k
        role: diagnostic
        bar: sanity
        reads_as: "Action-expert parameter coordinate count; if ~0 the path-based partition is broken"
compute:
  tier: cpu
  timeout_s: 900  # 2 arms x (jit compile ~2-4 min + 8 tiny steps + 8 evals) ~ 6-10 min; user cap 20 min
held_constant:
  - "bit-identical init: both arms from Pi0Config(dummy,dummy,action_dim=8,action_horizon=4,max_token_len=16,dtype=float32).create(jax.random.key(0))"
  - "one fixed random batch built from jax.random keys 1-4 exactly as in the PR's action_token_supervision_test.py"
  - "identical per-step noise/timestep keys: jax.random.fold_in(jax.random.key(7), step), same step index in both arms"
  - "identical 4 eval keys fold_in(key(101), i) for before/after losses in both arms"
  - "hand-built Adam lr=1e-3, b1=0.9, b2=0.999, eps=1e-8, weight_decay=0, 8 steps (REMYX_SMOKE=1 -> 2 steps), float32"
  - "action_token_supervision=False so no auxiliary CE term touches the VLM and the exact-zero criterion is meaningful"
  - "PR-only config kwargs (gradient_routing, action_token_supervision) are applied only when present in Pi0Config's dataclass fields; the baseline falls back to pre-change full routing instead of TypeError"
  - "parameter partition by the repo test convention: expert side = flat path contains '_1' or 'action_in_proj'/'action_out_proj'; everything else is VLM side"
avoid:
  - "no datasets, checkpoints, or network fetches; both transformer streams use the tiny dummy variants"
  - "no wall-clock timing on shared CPU; all reported quantities are deterministic weight/loss deltas"
  - "never use optax defaults or any weight_decay>0: decay moves zero-grad VLM weights and fakes a nonzero VLM delta under stop_vlm"
  - "never pass PR-only kwargs to Pi0Config unconditionally: on the pre-change baseline that is an unknown-kwarg TypeError, not a clean fallback"
  - "an 8-step dummy-scale run cannot test the paper's 25%/17%/40% training-outcome comparisons; do not read it as one"
  - "JAX backend only; the gradient_routing port in models_pytorch/pi0_pytorch.py is not exercised here"
  - "do not gate on loss_after below the initial-loss scale; 8 steps at lr=1e-3 are a mechanism probe, not convergence"
provenance:
  vlm_weight_l1_delta_stop_vlm: "user_guidance"
  vlm_weight_l1_delta_full: "user_guidance"
  expert_weight_l1_delta_stop_vlm: "user_guidance"
  expert_weight_l1_delta_full: "user_guidance"
  loss_drop_full: "user_guidance"
  loss_drop_stop_vlm: "user_guidance"
  loss_step0_gap: "inferred"
  loss_before_full: "user_guidance"
  loss_after_full: "user_guidance"
  loss_before_stop_vlm: "user_guidance"
  loss_after_stop_vlm: "user_guidance"
  vlm_param_count: "inferred"
  expert_param_count: "inferred"
  held_constant: "repo test convention from src/openpi/models/action_token_supervision_test.py"
  suite: "synthesized"
  compute: "user_guidance"
report:
  headline: "Insulation mechanism verified at dummy scale: stop_vlm leaves every VLM weight bit-identical while the action expert trains and the loss falls under both routings"
  findings:
    - "VLM-side L1 movement under stop_vlm is {vlm_weight_l1_delta_stop_vlm} (exactly 0 required) versus {vlm_weight_l1_delta_full} under full coupling"
    - "action-expert L1 movement is {expert_weight_l1_delta_stop_vlm} under stop_vlm and {expert_weight_l1_delta_full} under full"
    - "flow-matching loss falls by {loss_drop_full} under full and {loss_drop_stop_vlm} under stop_vlm, from initial values {loss_before_full} and {loss_before_stop_vlm}"
    - "initial losses agree across routings to {loss_step0_gap}, confirming the routing knob leaves forward outputs unchanged"
    - "the partition covers {vlm_param_count} VLM-side and {expert_param_count} action-expert coordinates"
  establishes:
    - "under stop_vlm the VLM trunk receives exactly zero gradient from the action expert's loss ({vlm_weight_l1_delta_stop_vlm}) while the expert keeps learning ({expert_weight_l1_delta_stop_vlm}, loss drop {loss_drop_stop_vlm})"
    - "under full routing the expert's loss does update the VLM ({vlm_weight_l1_delta_full}), so the knob switches real gradient flow rather than being a no-op"
  does_not_establish:
    - "which routing trains better on real data; the paper's 25%/17%/40% open-loop, closed-loop and manipulation deltas are out of scope at this scale"
    - "the PyTorch backend's gradient_routing port in models_pytorch/pi0_pytorch.py"
    - "action-token supervision, the PR's second knob and the other factor of the paper's 2x2 ablation"
  not_measured:
    - "open-loop trajectory-prediction error, closed-loop route progress, and robot manipulation success rate"
    - "throughput, memory, or wall-clock cost of the routing change"
  caveat: "The exact VLM zero holds only because this harness has no vision-language or action-token loss term; in the paper's insulated arms the VLM still trains, so this tests the routing mechanism, not the recipe's relative merit."
  next: "Run the paper's 2x2 (gradient routing x action-token supervision) on a real VLA dataset with open- and closed-loop evaluation to compare training outcomes."
```